# ORBIT: action-conditioned JEPA / действия и планирование / 动作与规划

**RU.** Минимальный воспроизводимый разбор новой модели. Она получает два изображения и три толчка, предсказывает три будущих вектора признаков. План выбирается по расстоянию до признаков картинки-цели — без координатного считывателя и без вызова симулятора во время выбора.

**EN.** A minimal executable walkthrough. Two observed images and three impulses produce three predicted future embeddings. Plan selection minimizes distance to an encoded goal image, without a coordinate probe or simulator calls inside the planner.

**中文.** 可运行的教学示例：模型根据两幅观测图像和三个动作预测未来特征。规划器在特征空间中选择最接近目标图像的方案，选择过程中不调用物理模拟器，也不使用坐标解码器。

Run from the repository root. Install `requirements-notebook.txt`. This notebook uses the released weights; it does not retrain them. / Запуск из корня проекта; здесь используются готовые веса. / 请从仓库根目录运行；此处加载已训练的权重。

In [1]:
from pathlib import Path
import itertools
import json
import numpy as np
import torch
from planning.model import ActionWorld, PLANS, load_encoder, load_predictor, rollout

assert Path("artifacts/action_jepa/predictor.npz").exists(), "Run from the repository root"
torch.set_num_threads(2)
encoder = load_encoder()
predictor = load_predictor("artifacts/action_jepa")
print("Frozen encoder parameters:", sum(p.numel() for p in encoder.parameters()))
print("Action predictor parameters:", sum(p.numel() for p in predictor.parameters()))
print("Candidate plans:", len(PLANS))
assert not any(p.requires_grad for p in encoder.parameters())

Frozen encoder parameters: 82632
Action predictor parameters: 23192
Candidate plans: 27


## 1. Observation and goal / наблюдение и цель / 观测与目标

**RU.** Симулятор создаёт учебную сцену и изображение достижимой цели. Состояние симулятора не передаётся планировщику. Его используем позже, чтобы проверить выбранное решение. Эта одна сцена — иллюстрация, не новый бенчмарк.

**EN.** We generate one observation and a reachable goal image. Simulator state stays outside the learner and is used afterward for evaluation. This single scene is an illustration, not a benchmark.

**中文.** 模拟器生成观测和一个可达目标的图像。规划器只接收图像；真实状态只用于事后验证。单个场景只是演示，不是新的基准测试。

In [2]:
world = ActionWorld(seed=41007)
images, sampled_actions, positions, velocity = world.sample(1)
context_images = images[:, :2]
reference_plan = torch.tensor([[1., 0., -1.]])
goal_position = rollout(positions[:, 1], velocity, reference_plan)[:, -1]
goal_image = world.render(goal_position, amplitude=torch.ones(1), noise=0)
print("Context image shape:", tuple(context_images.shape))
print("Goal image shape:", tuple(goal_image.shape))

Context image shape: (1, 2, 24, 24)
Goal image shape: (1, 24, 24)


## 2. Plan only in embedding space / планирование по признакам / 特征空间规划

**RU.** В этой функции нет обращения к физике, координатам или считывателю координат. Она получает только картинки, энкодер и обученный предиктор. Все 27 кандидатов сравниваются по одной целевой функции.

**EN.** This function has no access to physics, positions, or the coordinate probe. It compares all 27 candidates using the same learned feature-space objective.

**中文.** 此函数不访问物理规则、真实坐标或坐标解码器。它用同一个已学习特征空间的目标函数比较全部 27 个候选方案。

In [3]:
@torch.no_grad()
def choose_plan(context_images, goal_image, encoder, predictor):
    z = encoder(context_images.reshape(-1, 24, 24)).reshape(1, 2, 8)
    goal_z = encoder(goal_image)
    future_z = predictor(z.expand(len(PLANS), -1, -1), PLANS)
    costs = ((future_z[:, -1] - goal_z) / predictor.scale).square().mean(-1)
    best = int(costs.argmin())
    return PLANS[best].clone(), costs, future_z

chosen, costs, future_z = choose_plan(context_images, goal_image, encoder, predictor)
print("Chosen actions (-1 left, 0 coast, +1 right):", chosen.int().tolist())
print("Predicted shape [27 plans, 3 times, 8 features]:", tuple(future_z.shape))
for idx in costs.argsort()[:5]:
    i = int(idx)
    print(PLANS[i].int().tolist(), "feature cost =", round(float(costs[i]), 5))

Chosen actions (-1 left, 0 coast, +1 right): [1, -1, 1]
Predicted shape [27 plans, 3 times, 8 features]: (27, 3, 8)
[1, -1, 1] feature cost = 0.01787
[1, 0, -1] feature cost = 0.02357
[0, 1, 0] feature cost = 0.0269
[1, -1, 0] feature cost = 0.08243
[1, 0, 0] feature cost = 0.08449


## 3. Evaluate afterward / проверка после решения / 选择后验证

**RU.** Только теперь выполняем выбранные действия в настоящем симуляторе. Эталон с известной физикой имеет привилегированное точное состояние; его результат нельзя выдавать за возможности нейросети.

**EN.** Only now do we execute the chosen actions in the true simulator. The physics oracle has privileged exact state; its score is not the neural model's capability.

**中文.** 现在才在真实模拟器中执行已选动作。物理基准知道精确状态和规则，拥有额外信息，不能把它的成绩当成神经网络的能力。

In [4]:
with torch.no_grad():
    actual_end = rollout(positions[:, 1], velocity, chosen[None])[:, -1]
    distance = torch.linalg.vector_norm(actual_end - goal_position, dim=-1).item()
    all_true = rollout(positions[:, 1].expand(27, -1), velocity.expand(27, -1), PLANS)[:, -1]
    true_distances = torch.linalg.vector_norm(all_true - goal_position, dim=-1)
print("JEPA chosen plan, actual goal distance (pixels):", round(distance * 23, 4))
print("Goal reached within radius 0.06:", distance <= 0.06)
print("Privileged oracle distance (pixels):", round(float(true_distances.min()) * 23, 4))
print("Model score must be measured after execution, not from its own predicted cost.")

JEPA chosen plan, actual goal distance (pixels): 0.0
Goal reached within radius 0.06: True
Privileged oracle distance (pixels): 0.0
Model score must be measured after execution, not from its own predicted cost.


## 4. Published held-out evaluation / отдельная проверка / 独立测试

**RU.** Ниже загружается измеренный результат 1024 новых сцен. Один seed обучения, синтетический мир; это не сравнение с LLM и не универсальная робототехника. Энкодер взят из предыдущего проекта, обучен только новый предиктор действий.

**EN.** The released report uses 1,024 held-out scenes and a single training seed. Synthetic world only, not an LLM comparison or general robotics. The visual encoder is reused; only the new action predictor was trained for this release.

**中文.** 已发布的测试使用 1,024 个独立场景、一次训练随机种子。仅限合成环境，不是与 LLM 的比较，也不能证明通用机器人能力。本版复用视觉编码器，只训练新的动作预测器。

In [5]:
report = json.loads(Path("artifacts/action_jepa/evaluation.json").read_text())
for name in ["model", "random", "no_push", "shuffled_actions", "oracle"]:
    r = report[name]
    print(f"{name:18s} hit={100*r['hit_rate']['mean']:5.1f}%  distance={r['goal_distance_px']['mean']:.3f} px")
print("Held-out scenes:", report["n"])

model              hit= 67.8%  distance=0.942 px
random             hit= 19.5%  distance=4.322 px
no_push            hit= 23.7%  distance=3.411 px
shuffled_actions   hit= 21.7%  distance=4.290 px
oracle             hit=100.0%  distance=0.186 px
Held-out scenes: 1024


## Experiments / задания / 实验

**RU.** 1) Поменяйте цель, оставив наблюдения прежними: изменится ли план? 2) Переставьте значения действий только на входе предиктора: сохранится ли качество? 3) Повышайте шум на обоих входных изображениях и проверяйте *настоящее* попадание, а не только уверенность прогноза. 4) Для нового обучения: `python train_actions.py --out runs/my_action --steps 5000`. Не перезаписывайте опубликованные веса без резервной копии: их отпечаток задаёт версию рейтинга.

**EN.** Change the goal but keep the context; shuffle action meanings; increase observation noise and measure actual hits. Train separately with `python train_actions.py --out runs/my_action --steps 5000`. Back up release weights before replacing them; their fingerprint defines the leaderboard version.

**中文.** 保持观测不变并改变目标；打乱动作含义；增加观测噪声并测量实际命中率。可用上述命令在独立目录重新训练。替换发布权重前请备份，因为权重指纹决定排行榜版本。

**No universal intelligence or human-verified leaderboard is claimed.** / Нет заявления об универсальном интеллекте или защите рейтинга от ботов. / 不声称具有通用智能，也不声称排行榜能够验证参与者一定是人类。